# RetinaSafe — APTOS 2019 baseline (ResNet-50)

**Goal:** train a ResNet-50 to predict diabetic retinopathy severity (5-class ordinal: No DR / Mild / Moderate / Severe / Proliferative) on APTOS 2019. Report accuracy, per-class AUROC, macro AUROC, and **quadratic weighted kappa (QWK)** — the standard APTOS metric, which respects the ordinal structure of severity.

**Verified before writing this notebook:**
- Dataset attached at `/kaggle/input/datasets/mariaherrerot/aptos2019/`
- 3,662 RGB images across 80/10/10 image-disjoint train/val/test splits (2,930 / 366 / 366)
- All splits have labels (`diagnosis` column, integer 0–4)
- Class imbalance: ~50% class 0, ~5% class 3 — class-balanced sampling required
- Image dims range from 819×614 to 3388×2588 → resize uniformly to 224×224

**Caveats baked into the design:**
- **No patient IDs** in APTOS — splits are image-disjoint, not patient-disjoint. Documented as a known APTOS limitation; reviewers don't penalize you for this since it's a property of the dataset, not your methodology.
- **No black-border crop** in the baseline — plain Resize to 224. Adding a smart-crop step would likely improve performance another 1–2 pt but is left for a follow-up.

**How to run:** cells 1–6 work on CPU (sanity-check first), then enable GPU T4 + Save & Run All for training. Total runtime ~30–45 min on T4 (much faster than Kermany because the dataset is 23× smaller).

## Cell 1 — Imports + GPU check

In [ ]:
import os, sys, json, time, random, pathlib
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import (roc_auc_score, accuracy_score, confusion_matrix,
                              cohen_kappa_score, classification_report)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ---------- Paths ----------
ROOT = "/kaggle/input/datasets/mariaherrerot/aptos2019"
TRAIN_CSV = os.path.join(ROOT, "train_1.csv")
VAL_CSV   = os.path.join(ROOT, "valid.csv")
TEST_CSV  = os.path.join(ROOT, "test.csv")
TRAIN_IMG = os.path.join(ROOT, "train_images", "train_images")
VAL_IMG   = os.path.join(ROOT, "val_images",   "val_images")
TEST_IMG  = os.path.join(ROOT, "test_images",  "test_images")

WORK = pathlib.Path("/kaggle/working")
RESULTS = WORK / "results"
RESULTS.mkdir(exist_ok=True)
CKPT = RESULTS / "best.pt"

print("Torch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for p in [TRAIN_CSV, VAL_CSV, TEST_CSV, TRAIN_IMG, VAL_IMG, TEST_IMG]:
    assert os.path.exists(p), f"Missing: {p}"
print("All paths verified.")

## Cell 2 — Build a unified index

One CSV with `[id_code, split, diagnosis, abs_path]` so the rest of the notebook reads from one source.

In [ ]:
def build_split_index(csv, img_dir, split_name):
    df = pd.read_csv(csv)
    df["split"] = split_name
    df["abs_path"] = df["id_code"].apply(lambda x: os.path.join(img_dir, f"{x}.png"))
    return df

idx = pd.concat([
    build_split_index(TRAIN_CSV, TRAIN_IMG, "train"),
    build_split_index(VAL_CSV,   VAL_IMG,   "val"),
    build_split_index(TEST_CSV,  TEST_IMG,  "test"),
], ignore_index=True)

# Validate every image exists on disk
missing = ~idx["abs_path"].apply(os.path.exists)
print(f"Missing image files: {int(missing.sum())}/{len(idx)}")
if missing.any():
    print(idx.loc[missing, ['split', 'id_code']].head())
assert not missing.any(), "Some images are missing — investigate before training"

INDEX_CSV = WORK / "aptos_index.csv"
idx.to_csv(INDEX_CSV, index=False)
print(f"\nWrote {INDEX_CSV} ({len(idx):,} rows)")

# Distribution check
print("\nClass distribution by split:")
print(idx.groupby(["split", "diagnosis"]).size().unstack(fill_value=0))

## Cell 3 — Dataset + transforms

In [ ]:
CLASSES = ["No_DR", "Mild", "Moderate", "Severe", "Proliferative"]
N_CLASSES = len(CLASSES)
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def train_tf(size=IMG_SIZE):
    return transforms.Compose([
        transforms.Resize((size + 32, size + 32)),
        transforms.RandomCrop(size),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),   # fundus has no canonical up
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

def eval_tf(size=IMG_SIZE):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class APTOSDataset(Dataset):
    def __init__(self, index_csv, split, transform=None):
        full = pd.read_csv(index_csv)
        self.df = full[full["split"] == split].reset_index(drop=True)
        self.transform = transform
        self.labels = self.df["diagnosis"].astype(int).to_numpy()

    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["abs_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx]), row["id_code"]


print("Dataset class + transforms defined.")

## Cell 4 — Loaders + sanity batch

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2

train_ds = APTOSDataset(INDEX_CSV, "train", transform=train_tf())
val_ds   = APTOSDataset(INDEX_CSV, "val",   transform=eval_tf())
test_ds  = APTOSDataset(INDEX_CSV, "test",  transform=eval_tf())
print(f"train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}")

# Class-balanced sampler for training
class_counts = np.bincount(train_ds.labels, minlength=N_CLASSES)
sample_weights = 1.0 / class_counts[train_ds.labels]
sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(train_ds),
    replacement=True,
)
print(f"Class counts (train): {dict(zip(CLASSES, class_counts.tolist()))}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# Sanity batch
imgs, labels, _ = next(iter(train_loader))
print(f"\nBatch: shape={imgs.shape}  dtype={imgs.dtype}  range=[{imgs.min():.2f}, {imgs.max():.2f}]")
print(f"Sampler balance (should be ~roughly equal across classes):")
for cls_idx, n in sorted(Counter(labels.tolist()).items()):
    print(f"  {CLASSES[cls_idx]:13s}: {n}")

## Cell 5 — Build ResNet-50 (ImageNet pretrained)

In [ ]:
def build_resnet50(num_classes=N_CLASSES, dropout=0.3):
    weights = models.ResNet50_Weights.IMAGENET1K_V2
    m = models.resnet50(weights=weights)
    in_feat = m.fc.in_features
    m.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_feat, num_classes),
    )
    return m

model = build_resnet50().to(DEVICE)
print(f"ResNet-50 built. Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

# Sanity forward pass
model.eval()
with torch.no_grad():
    out = model(imgs.to(DEVICE))
print(f"Forward pass OK. Output shape: {out.shape}  (expected: [{BATCH_SIZE}, {N_CLASSES}])")

## Cell 6 — Training loop

**Setup:** AdamW (lr=3e-4 — higher than Kermany because dataset is smaller), cosine schedule with 2-epoch warmup, AMP, **early stop on val QWK** (the metric APTOS papers report). Class-weighted CE on top of the balanced sampler.

**Why QWK as the monitor metric:** DR severity is ordinal (0→4 is disease progression). QWK penalizes by squared distance — misclassifying class 0 as class 1 costs ~1, class 0 as class 4 costs ~16. Plain accuracy treats all errors equally, which is wrong for DR.

In [ ]:
EPOCHS = 30
LR = 3e-4
WD = 1e-4
WARMUP_EPOCHS = 2
PATIENCE = 6

@torch.no_grad()
def evaluate(model, loader, device, return_logits=False):
    model.eval()
    losses, all_logits, all_labels = [], [], []
    crit = nn.CrossEntropyLoss()
    for x, y, _ in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(x)
            loss = crit(logits, y)
        losses.append(loss.item() * x.size(0))
        all_logits.append(logits.float().cpu())
        all_labels.append(y.cpu())
    n = sum(t.size(0) for t in all_labels)
    avg_loss = sum(losses) / n
    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    probs = F.softmax(logits, dim=1).numpy()
    y_true = labels.numpy()
    y_pred = probs.argmax(axis=1)
    # Macro AUROC (one-vs-rest, on classes present)
    aurocs = []
    for c in range(N_CLASSES):
        y_bin = (y_true == c).astype(int)
        if y_bin.sum() == 0 or y_bin.sum() == len(y_bin): continue
        aurocs.append(roc_auc_score(y_bin, probs[:, c]))
    macro_auroc = float(np.mean(aurocs)) if aurocs else float('nan')
    acc = accuracy_score(y_true, y_pred)
    qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    if return_logits:
        return avg_loss, macro_auroc, acc, qwk, logits, labels
    return avg_loss, macro_auroc, acc, qwk

def cosine_warmup_lr(epoch, total, warmup, base):
    if epoch < warmup:
        return base * (epoch + 1) / warmup
    p = (epoch - warmup) / max(1, total - warmup)
    return base * 0.5 * (1 + np.cos(np.pi * p))


cw = torch.tensor(1.0 / class_counts, dtype=torch.float32)
cw = cw / cw.mean()
criterion = nn.CrossEntropyLoss(weight=cw.to(DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

best_qwk = -1e9
best_epoch = -1
no_improve = 0
history = []

for epoch in range(EPOCHS):
    lr_now = cosine_warmup_lr(epoch, EPOCHS, WARMUP_EPOCHS, LR)
    for g in optimizer.param_groups: g["lr"] = lr_now

    model.train()
    t0 = time.time()
    running = 0.0; n_seen = 0
    for x, y, _ in train_loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        running += loss.item() * x.size(0); n_seen += x.size(0)
    train_loss = running / n_seen

    val_loss, val_auroc, val_acc, val_qwk = evaluate(model, val_loader, DEVICE)
    dur = time.time() - t0
    history.append({"epoch": epoch, "lr": lr_now, "train_loss": train_loss,
                    "val_loss": val_loss, "val_auroc": val_auroc,
                    "val_acc": val_acc, "val_qwk": val_qwk, "seconds": dur})
    print(f"Epoch {epoch+1:02d}/{EPOCHS}  lr={lr_now:.2e}  "
          f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"val_auroc={val_auroc:.4f}  val_acc={val_acc:.4f}  val_qwk={val_qwk:.4f}  ({dur:.0f}s)")

    if val_qwk > best_qwk:
        best_qwk = val_qwk; best_epoch = epoch; no_improve = 0
        torch.save({"model_state": model.state_dict(), "epoch": epoch,
                    "val_qwk": val_qwk, "val_auroc": val_auroc, "classes": CLASSES}, CKPT)
        print(f"  -> new best (val_qwk={val_qwk:.4f}), saved {CKPT}")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

pd.DataFrame(history).to_csv(RESULTS / "training_history.csv", index=False)
print(f"\nBest val_qwk={best_qwk:.4f} at epoch {best_epoch+1}")

## Cell 7 — Test eval with bootstrap 95% CIs

In [ ]:
state = torch.load(CKPT, map_location=DEVICE)
model.load_state_dict(state["model_state"])
print(f"Loaded checkpoint epoch {state['epoch']+1}  val_qwk={state['val_qwk']:.4f}")

test_loss, test_auroc, test_acc, test_qwk, test_logits, test_labels = evaluate(
    model, test_loader, DEVICE, return_logits=True
)
test_probs = F.softmax(test_logits, dim=1).numpy()
y_true = test_labels.numpy()
y_pred = test_probs.argmax(axis=1)

per_class_auroc = {}
for i, c in enumerate(CLASSES):
    y_bin = (y_true == i).astype(int)
    per_class_auroc[c] = float(roc_auc_score(y_bin, test_probs[:, i])) if y_bin.sum() else None

cm = confusion_matrix(y_true, y_pred, labels=list(range(N_CLASSES)))

def bootstrap_ci(probs, y_true, n_boot=1000, alpha=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    accs, qwks, macro_aurocs = [], [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        pb, yb = probs[idx], y_true[idx]
        accs.append(accuracy_score(yb, pb.argmax(axis=1)))
        qwks.append(cohen_kappa_score(yb, pb.argmax(axis=1), weights='quadratic'))
        aurocs = []
        for c in range(N_CLASSES):
            ybin = (yb == c).astype(int)
            if ybin.sum() == 0 or ybin.sum() == len(ybin): continue
            aurocs.append(roc_auc_score(ybin, pb[:, c]))
        if aurocs: macro_aurocs.append(np.mean(aurocs))
    def ci(v):
        v = np.asarray(v)
        return [float(np.quantile(v, alpha/2)), float(np.quantile(v, 1-alpha/2))]
    return ci(accs), ci(qwks), ci(macro_aurocs)

acc_ci, qwk_ci, auroc_ci = bootstrap_ci(test_probs, y_true)

results = {
    "test_n": int(len(y_true)),
    "test_loss": float(test_loss),
    "test_accuracy": float(test_acc),
    "test_accuracy_ci95": acc_ci,
    "test_qwk": float(test_qwk),
    "test_qwk_ci95": qwk_ci,
    "test_auroc_macro": float(test_auroc),
    "test_auroc_macro_ci95": auroc_ci,
    "test_auroc_per_class": per_class_auroc,
    "confusion_matrix": cm.tolist(),
    "classes": CLASSES,
    "best_epoch": int(state["epoch"] + 1),
    "best_val_qwk": float(state["val_qwk"]),
}
with open(RESULTS / "test_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n=== Test results ===")
print(json.dumps(results, indent=2))

print("\nConfusion matrix (rows=true, cols=pred):")
print("                 " + "  ".join(f"{c:>14s}" for c in CLASSES))
for i, c in enumerate(CLASSES):
    print(f"  {c:>13s}  " + "  ".join(f"{cm[i,j]:>14d}" for j in range(N_CLASSES)))

print("\nClassification report (per-class precision/recall):")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))

## Cell 8 — Persist artifacts off Kaggle

In [ ]:
import shutil
zip_path = WORK / "aptos_baseline_results.zip"
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(zip_path.with_suffix(""), "zip", root_dir=str(RESULTS))
print(f"Wrote {zip_path}  ({zip_path.stat().st_size/1e6:.1f} MB)")
print("Download from Notebook page → Output tab → aptos_baseline_results.zip")